# HPD 1 — Entregable: Pipeline de embeddings y búsqueda semántica

<figure>
<a
href="https://colab.research.google.com/github/Adamychen/m10_quarto/blob/main/notebooks/evaluables/hpd1-entregable.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open In Colab</figcaption>
</figure>

> **Peso en la nota:** 7.5 %
>
> **Plazo:** por determinar.
>
> **Entrega:** Notebook `.ipynb` ejecutado con todas las celdas
> completas. Cada ejercicio especifica qué variable debe contener el
> resultado para la corrección automática.

> **Cómo se corrige**
>
> Cada ejercicio pide que asignes el resultado a una variable con un
> nombre concreto (`eval1_estrategia`, `eval2_recall`, `eval3_busqueda`,
> `eval4_justificacion`). El script de corrección ejecutará tu notebook
> e inspeccionará esas variables.
>
> Además, la variable `CAMINO` debe estar definida al inicio del
> notebook.

In [1]:
CAMINO = "A"  # Cambiar a "B" o "C" según tu elección

## Setup

In [2]:
!pip install -q chromadb sentence-transformers fpdf2 matplotlib pandas langchain langchain-community langchain-text-splitters pypdf python-dotenv datasets

In [3]:
import os, tempfile, shutil
"""
Setup compartido para la API key de Llamus en todos los notebooks.
Inyectado en cada .qmd/.ipynb vía scripts/sync_setup_keys.py (pre-render).

Orden de resolución:
  1. .env local (python-dotenv)
  2. os.environ ya configurado externamente
  3. Google Colab userdata: secret "LLAMUS_API_KEY" (preferente) o "LLM_API_KEY"

Las celdas posteriores pueden leer la clave con os.environ.get("LLM_API_KEY").
"""

import os


def _load_llm_api_key():
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

    if not os.getenv("LLM_API_KEY"):
        try:
            from google.colab import userdata
            for name in ("LLAMUS_API_KEY", "LLM_API_KEY"):
                try:
                    value = userdata.get(name)
                    if value:
                        os.environ["LLM_API_KEY"] = value
                        break
                except Exception:
                    continue
        except ImportError:
            pass


_load_llm_api_key()

if os.getenv("LLM_API_KEY"):
    print("✓ LLM_API_KEY cargada.")
else:
    print("⚠️  LLM_API_KEY no encontrada.")
    print("    • Local: crea un archivo .env con LLM_API_KEY=tu_clave")
    print("    • Colab: añade un Secret 🔑 con nombre 'LLAMUS_API_KEY'")


os.environ["TOKENIZERS_PARALLELISM"] = "false"
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fpdf import FPDF
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 100
print("Librerías cargadas correctamente.")

✓ LLM_API_KEY cargada.
Librerías cargadas correctamente.

In [4]:
# ------------------------------------------------------------------ Parámetros compartidos
MODEL_NAME = "intfloat/e5-small-v2"
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

BASE_DIR = tempfile.mkdtemp()
CORPUS_DIR = os.path.join(BASE_DIR, "corpus")
os.makedirs(CORPUS_DIR, exist_ok=True)
print(f"Directorio temporal: {BASE_DIR}")

# ------------------------------------------------------------------ Modelo de embeddings
model = SentenceTransformer(MODEL_NAME)
print(f"Modelo cargado: {MODEL_NAME} ({model.get_embedding_dimension()} dimensiones)")

# ------------------------------------------------------------------ Chroma
client = chromadb.EphemeralClient()

Directorio temporal: /var/folders/5q/dq_s1n552d37rqhcc02ppsn00000gn/T/tmpdlwx9wta

Modelo cargado: intfloat/e5-small-v2 (384 dimensiones)

In [5]:
class PDF(FPDF):
    pass

def crear_pdf(nombre: str, titulo: str, contenido: str) -> None:
    pdf = PDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    titulo_safe = titulo.replace("—", "-").replace("€", "EUR").replace("…", "...")
    pdf.cell(0, 10, titulo_safe, new_x="LMARGIN", new_y="NEXT")
    pdf.ln(5)
    pdf.set_font("Helvetica", "", 11)
    contenido_safe = contenido.replace("—", "-").replace("€", "EUR").replace("…", "...")
    pdf.multi_cell(0, 6, contenido_safe)
    pdf.output(os.path.join(CORPUS_DIR, nombre))
    print(f"  ✅ {nombre}")

print("Helper PDF listo.")

Helper PDF listo.

## Camino A — Finanzas

**Contexto:** Trabajas como Data Scientist en una consultora financiera.
Tu cliente te pide un sistema de búsqueda semántica sobre documentación
financiera real del benchmark FIQA (Financial Question Answering), un
benchmark del ecosistema BEIR. El sistema debe permitir a los analistas
buscar información por concepto, no por palabra clave.

### Corpus — Documentos financieros reales (FIQA)

``` python
if CAMINO == "A":
    from datasets import load_dataset

    fiqa_corpus = load_dataset("BeIR/fiqa", "corpus", split="corpus")
    fiqa_queries = load_dataset("BeIR/fiqa", "queries", split="queries")
    fiqa_qrels = load_dataset("BeIR/fiqa-qrels", split="train")

    # Filtrar corpus a documentos que tengan qrels (ground truth real)
    qrel_cids = {str(cid) for cid in fiqa_qrels["corpus-id"]}
    fiqa_filtered = fiqa_corpus.filter(lambda x: x["_id"] in qrel_cids)
    fiqa_subset = fiqa_filtered.select(range(30))
    print(f"FIQA corpus cargado: {len(fiqa_subset)} documentos financieros reales con qrels")

    docs_a = []
    for doc in fiqa_subset:
        docs_a.append(Document(
            page_content=doc["text"],
            metadata={"source": f"fiqa_{doc['_id']}.txt", "fiqa_id": doc["_id"]}
        ))

    subset_ids = {int(doc["_id"]) for doc in fiqa_subset}
    qid_to_corpus_ids = {}
    for qrel in fiqa_qrels:
        if qrel["corpus-id"] in subset_ids:
            qid = str(qrel["query-id"])
            if qid not in qid_to_corpus_ids:
                qid_to_corpus_ids[qid] = []
            qid_to_corpus_ids[qid].append(str(qrel["corpus-id"]))

    selected_qids = list(qid_to_corpus_ids.keys())[:5]
    queries_map = {}
    for q in fiqa_queries:
        if q["_id"] in selected_qids:
            queries_map[q["_id"]] = q["text"]

    gt_a_raw = {}
    for qid in selected_qids:
        gt_a_raw[queries_map[qid]] = qid_to_corpus_ids[qid]

    print(f"Queries de evaluación: {len(selected_qids)}")
    print(f"Documentos en corpus: {len(docs_a)}")
else:
    print("Camino A no seleccionado — saltando generación.")
```

    FIQA corpus cargado: 30 documentos financieros reales con qrels
    Queries de evaluación: 5
    Documentos en corpus: 30

#### Ejercicio 1 — Estrategia de chunking (2 ptos)

Describe qué estrategia de chunking eliges (chunk por párrafos
vs. tamaño fijo con solapamiento) y por qué es adecuada para documentos
financieros reales del benchmark FIQA. Justifica tu respuesta en
términos de:

- Coherencia semántica de los chunks
- Capacidad de recuperar información con consultas sobre magnitudes
  concretas (ej. “ratio de apalancamiento”, “ROE”, “morosidad”)
- Ventajas del solapamiento entre chunks en este dominio

Asigna tu respuesta a `eval1_estrategia`.

``` python
if CAMINO == "A":
    eval1_estrategia = """Escribir aquí la justificación. Por ejemplo:

    Elijo chunking por tamaño fijo con RecursiveCharacterTextSplitter (chunk_size=300, overlap=50)
    porque los documentos financieros de FIQA contienen párrafos densos con múltiples métricas. El
    solapamiento asegura que magnitudes concretas no se pierdan en los cortes entre chunks. La
    estrategia por párrafos no es adecuada porque los párrafos de FIQA varían en longitud y pueden
    mezclar varios conceptos en uno solo."""
    print("✅ eval1_estrategia asignada.")
else:
    eval1_estrategia = None
```

    ✅ eval1_estrategia asignada.

#### Ejercicio 2 — Indexación y recall@k (2.5 ptos)

Genera los embeddings de los documentos FIQA, construye la colección en
Chroma y evalúa con recall@k usando el ground truth derivado de las
anotaciones reales del benchmark.

``` python
if CAMINO == "A":
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks_a = splitter.split_documents(docs_a)
    print(f"Chunks generados: {len(chunks_a)}")

    collection_a = client.get_or_create_collection(
        name="hpd1_finanzas",
        metadata={"hnsw:space": "cosine"}
    )

    fiqa_to_chunk_ids = {}
    for i, chunk in enumerate(chunks_a):
        embedding = model.encode(chunk.page_content).tolist()
        fiqa_id = chunk.metadata.get("fiqa_id", "")
        cid = f"fin_{i:04d}"
        collection_a.add(
            ids=[cid],
            embeddings=[embedding],
            documents=[chunk.page_content],
            metadatas=[{"fiqa_id": fiqa_id, "idx": i}]
        )
        if fiqa_id not in fiqa_to_chunk_ids:
            fiqa_to_chunk_ids[fiqa_id] = []
        fiqa_to_chunk_ids[fiqa_id].append(cid)
    print(f"Indexación completada: {collection_a.count()} vectores")

    gt_a = {}
    for q_text, corpus_ids in gt_a_raw.items():
        chunk_ids = []
        for cid in corpus_ids:
            chunk_ids.extend(fiqa_to_chunk_ids.get(cid, []))
        gt_a[q_text] = chunk_ids
    print(f"Ground truth construido: {len(gt_a)} queries con anotaciones reales de FIQA")

    def recall_at_k(retrieved_ids, relevant_ids, k):
        if len(relevant_ids) == 0:
            return 0.0
        top_k = retrieved_ids[:k]
        hits = sum(1 for rid in relevant_ids if rid in top_k)
        return hits / len(relevant_ids)

    def buscar(collection, query, k=5):
        qe = model.encode(query).tolist()
        return collection.query(query_embeddings=[qe], n_results=k)

    eval2_recall = {}
    for k in [1, 3, 5]:
        recalls = []
        for q, relevantes in gt_a.items():
            res = buscar(collection_a, q, k=k)
            retrieved = res["ids"][0]
            r = recall_at_k(retrieved, relevantes, k)
            recalls.append(r)
        eval2_recall[f"recall@{k}"] = round(np.mean(recalls), 4)

    print("📊 recall@k sobre FIQA corpus (ground truth real):")
    for k_str, v in eval2_recall.items():
        print(f"   {k_str}: {v:.4f}")
else:
    eval2_recall = None
```

    Chunks generados: 170
    Indexación completada: 170 vectores
    Ground truth construido: 5 queries con anotaciones reales de FIQA
    📊 recall@k sobre FIQA corpus (ground truth real):
       recall@1: 0.1871
       recall@3: 0.4757
       recall@5: 0.5443

#### Ejercicio 3 — Búsqueda semántica (1.5 ptos)

Ejecuta 3 consultas de búsqueda semántica sobre el corpus FIQA y muestra
los resultados (top-3 chunks con su distancia coseno). Las consultas son
preguntas reales del benchmark.

Asigna los resultados (lista de diccionarios con query, top-3 chunks y
distancias) a `eval3_busqueda`.

``` python
if CAMINO == "A":
    consultas_a = list(gt_a.keys())[:3]

    eval3_busqueda = []
    for q in consultas_a:
        res = buscar(collection_a, q, k=3)
        resultados = []
        for i in range(len(res["ids"][0])):
            resultados.append({
                "id": res["ids"][0][i],
                "distancia": round(res["distances"][0][i], 4),
                "texto": res["documents"][0][i][:200],
                "fuente": res["metadatas"][0][i].get("fiqa_id", ""),
            })
        eval3_busqueda.append({"query": q, "resultados": resultados})

    for r in eval3_busqueda:
        print(f"\n🔍 {r['query']}")
        for res in r["resultados"]:
            print(f"   [{res['distancia']:.4f}] {res['texto'][:100]}...")
    print("\n✅ eval3_busqueda asignada.")
else:
    eval3_busqueda = None
```

    🔍 How Long Can It Take For a Check I Write to Clear on My Account?
       [0.1376] . However, a check is a debt instrument, and you cannot close the account until it clears, or until ...
       [0.1411] "There's nothing you can do. If he has indeed deposited the check, it would appear on your account f...
       [0.1428] . If he claims that he deposited the check, ask it in writing and have your bank (or the bank where ...

    🔍 Freehold and Leasehold for Pub/Bar?
       [0.1287] . In this case, you may be purchasing the business of the pub from a pub company, but the building t...
       [0.1382] . mentioned in his comment, where you're purchasing the building and business outright, but the land...
       [0.1403] "In the strictest sense of the words, Freehold and leasehold mean what you think they do. Freehold i...

    🔍 How does a Non US citizen gain SEC Accredited Investor Status?
       [0.1255] Here are the SEC requirements: The federal securities laws define the term accredited investor in   ...
       [0.1544] . learn accounting, the basic at least open a  broker simulating account online ( you will play with...
       [0.1602] . Depending on your field you may have some inside knowledge so even if you would took a position wi...

    ✅ eval3_busqueda asignada.

#### Ejercicio 4 — Justificación del modelo (1.5 ptos)

Justifica por qué `intfloat/e5-small-v2` es una buena elección para este
pipeline. Debes basarte en los resultados de MTEB para las tareas
BIOSSES y STS12, y explicar cómo se relacionan con la calidad esperada
en búsqueda semántica sobre un corpus financiero real como FIQA.

Asigna tu respuesta a `eval4_justificacion`.

``` python
if CAMINO == "A":
    eval4_justificacion = """Escribir aquí la justificación. Por ejemplo:

    e5-small-v2 obtiene un Spearman de 0.894 en BIOSSES (similitud semántica biomédica) y
    0.774 en STS12 (dominio general). Aunque FIQA es un benchmark financiero, el alto
    rendimiento en STS12 indica que el modelo captura bien relaciones semánticas en texto
    narrativo, que es el formato dominante en los documentos de FIQA. Con 33M de parámetros,
    es eficiente para ejecutarse en local sin GPU, adecuado para un pipeline docente."""
    print("✅ eval4_justificacion asignada.")
else:
    eval4_justificacion = None
```

    ✅ eval4_justificacion asignada.

## Camino B — Salud

**Contexto:** Trabajas como Data Scientist en un laboratorio de
investigación biomédica. Necesitas indexar literatura científica real
del benchmark NFCorpus (biomedical information retrieval), un benchmark
del ecosistema BEIR. El sistema debe permitir a los investigadores
buscar por concepto, no solo por palabras clave.

### Corpus — Documentos biomédicos reales (NFCorpus)

``` python
if CAMINO == "B":
    from datasets import load_dataset

    nf_corpus = load_dataset("BeIR/nfcorpus", "corpus", split="corpus")
    nf_queries = load_dataset("BeIR/nfcorpus", "queries", split="queries")
    nf_qrels = load_dataset("BeIR/nfcorpus-qrels", split="train")

    qrel_cids = set(nf_qrels["corpus-id"])
    nf_filtered = nf_corpus.filter(lambda x: x["_id"] in qrel_cids)
    nf_subset = nf_filtered.select(range(30))
    print(f"NFCorpus cargado: {len(nf_subset)} documentos biomédicos reales con qrels")

    docs_b = []
    for doc in nf_subset:
        text = f"Title: {doc['title']}\n\n{doc['text']}"
        docs_b.append(Document(
            page_content=text,
            metadata={"source": f"nf_{doc['_id']}.txt", "nf_id": doc["_id"]}
        ))

    subset_ids = set(nf_subset["_id"])
    qid_to_corpus_ids_b = {}
    for qrel in nf_qrels:
        if qrel["corpus-id"] in subset_ids:
            qid = qrel["query-id"]
            if qid not in qid_to_corpus_ids_b:
                qid_to_corpus_ids_b[qid] = []
            qid_to_corpus_ids_b[qid].append(qrel["corpus-id"])

    selected_qids_b = list(qid_to_corpus_ids_b.keys())[:5]
    queries_map_b = {}
    for q in nf_queries:
        if q["_id"] in selected_qids_b:
            queries_map_b[q["_id"]] = q["text"]

    gt_b_raw = {}
    for qid in selected_qids_b:
        gt_b_raw[queries_map_b[qid]] = qid_to_corpus_ids_b[qid]

    print(f"Queries de evaluación: {len(selected_qids_b)}")
    print(f"Documentos en corpus: {len(docs_b)}")
else:
    print("Camino B no seleccionado — saltando generación.")
```

    Camino B no seleccionado — saltando generación.

#### Ejercicio 1 — Estrategia de chunking (2 ptos)

Describe qué estrategia de chunking eliges (chunk por párrafos
vs. tamaño fijo con solapamiento) y por qué es adecuada para documentos
biomédicos reales del benchmark NFCorpus. Justifica tu respuesta en
términos de:

- Coherencia semántica de los chunks
- Capacidad de recuperar información con consultas sobre conceptos
  específicos (ej. “AUC en detección de nódulos”, “eficacia del
  compuesto”, “efectos adversos”)
- Ventajas del solapamiento entre chunks en este dominio

Asigna tu respuesta a `eval1_estrategia`.

``` python
if CAMINO == "B":
    eval1_estrategia = """Escribir aquí la justificación. Por ejemplo:

    Elijo chunking por tamaño fijo con RecursiveCharacterTextSplitter (chunk_size=300, overlap=50)
    porque los artículos biomédicos de NFCorpus contienen párrafos densos con múltiples métricas (p-valores,
    AUC, tamaños muestrales). El solapamiento evita que conceptos clave no se pierdan en los cortes. La
    estrategia por párrafos no es adecuada porque los párrafos de los artículos científicos varían mucho
    en longitud y pueden mezclar varios resultados experimentales."""
    print("✅ eval1_estrategia asignada.")
else:
    eval1_estrategia = None
```

#### Ejercicio 2 — Indexación y recall@k (2.5 ptos)

Genera los embeddings de los documentos NFCorpus, construye la colección
en Chroma y evalúa con recall@k usando el ground truth derivado de las
anotaciones reales del benchmark.

``` python
if CAMINO == "B":
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks_b = splitter.split_documents(docs_b)
    print(f"Chunks generados: {len(chunks_b)}")

    collection_b = client.get_or_create_collection(
        name="hpd1_salud",
        metadata={"hnsw:space": "cosine"}
    )

    nf_to_chunk_ids = {}
    for i, chunk in enumerate(chunks_b):
        embedding = model.encode(chunk.page_content).tolist()
        nf_id = chunk.metadata.get("nf_id", "")
        cid = f"sal_{i:04d}"
        collection_b.add(
            ids=[cid],
            embeddings=[embedding],
            documents=[chunk.page_content],
            metadatas=[{"nf_id": nf_id, "idx": i}]
        )
        if nf_id not in nf_to_chunk_ids:
            nf_to_chunk_ids[nf_id] = []
        nf_to_chunk_ids[nf_id].append(cid)
    print(f"Indexación completada: {collection_b.count()} vectores")

    gt_b = {}
    for q_text, corpus_ids in gt_b_raw.items():
        chunk_ids = []
        for cid in corpus_ids:
            chunk_ids.extend(nf_to_chunk_ids.get(cid, []))
        gt_b[q_text] = chunk_ids
    print(f"Ground truth construido: {len(gt_b)} queries con anotaciones reales de NFCorpus")

    def recall_at_k(retrieved_ids, relevant_ids, k):
        if len(relevant_ids) == 0:
            return 0.0
        top_k = retrieved_ids[:k]
        hits = sum(1 for rid in relevant_ids if rid in top_k)
        return hits / len(relevant_ids)

    def buscar(collection, query, k=5):
        qe = model.encode(query).tolist()
        return collection.query(query_embeddings=[qe], n_results=k)

    eval2_recall = {}
    for k in [1, 3, 5]:
        recalls = []
        for q, relevantes in gt_b.items():
            res = buscar(collection_b, q, k=k)
            retrieved = res["ids"][0]
            r = recall_at_k(retrieved, relevantes, k)
            recalls.append(r)
        eval2_recall[f"recall@{k}"] = round(np.mean(recalls), 4)

    print("📊 recall@k sobre NFCorpus corpus (ground truth real):")
    for k_str, v in eval2_recall.items():
        print(f"   {k_str}: {v:.4f}")
else:
    eval2_recall = None
```

#### Ejercicio 3 — Búsqueda semántica (1.5 ptos)

Ejecuta 3 consultas de búsqueda semántica sobre el corpus NFCorpus y
muestra los resultados (top-3 chunks con su distancia coseno). Las
consultas son preguntas reales del benchmark.

Asigna los resultados a `eval3_busqueda`.

``` python
if CAMINO == "B":
    consultas_b = list(gt_b.keys())[:3]

    eval3_busqueda = []
    for q in consultas_b:
        res = buscar(collection_b, q, k=3)
        resultados = []
        for i in range(len(res["ids"][0])):
            resultados.append({
                "id": res["ids"][0][i],
                "distancia": round(res["distances"][0][i], 4),
                "texto": res["documents"][0][i][:200],
                "fuente": res["metadatas"][0][i].get("nf_id", ""),
            })
        eval3_busqueda.append({"query": q, "resultados": resultados})

    for r in eval3_busqueda:
        print(f"\n🔍 {r['query']}")
        for res in r["resultados"]:
            print(f"   [{res['distancia']:.4f}] {res['texto'][:100]}...")
    print("\n✅ eval3_busqueda asignada.")
else:
    eval3_busqueda = None
```

#### Ejercicio 4 — Justificación del modelo (1.5 ptos)

Justifica por qué `intfloat/e5-small-v2` es una buena elección para este
pipeline. Debes basarte en los resultados de MTEB para las tareas
BIOSSES y STS12, y explicar cómo se relacionan con la calidad esperada
en búsqueda semántica sobre un corpus biomédico real como NFCorpus.

Asigna tu respuesta a `eval4_justificacion`.

``` python
if CAMINO == "B":
    eval4_justificacion = """Escribir aquí la justificación. Por ejemplo:

    e5-small-v2 obtiene un Spearman de 0.894 en BIOSSES, un benchmark de similitud textual
    biomédica. Este resultado es directamente relevante para nuestro corpus NFCorpus de artículos de
    investigación clínica: indica que el modelo alinea bien textos con terminología médica
    y expresiones científicas. El 0.774 en STS12 (dominio general) confirma que mantiene
    buena calidad en texto narrativo. Con 33M de parámetros, es lo suficientemente ligero
    para ejecutarse en entornos como Colab sin GPU."""
    print("✅ eval4_justificacion asignada.")
else:
    eval4_justificacion = None
```

## Camino C — E-commerce

**Contexto:** Trabajas como Data Scientist en un marketplace digital. Tu
equipo necesita un sistema de búsqueda semántica sobre productos reales
de Amazon del benchmark ESCI (Amazon Shopping Queries). El sistema debe
permitir a los compradores encontrar productos por concepto y necesidad,
no solo por palabras clave.

### Corpus — Productos reales de Amazon (ESCI)

``` python
if CAMINO == "C":
    import urllib.request, gzip, io, json

    url = "https://huggingface.co/datasets/spacemanidol/ESCI-product-search/resolve/main/dev.jsonl.gz"
    response = urllib.request.urlopen(url)
    buf = io.BufferedReader(response)
    gz = gzip.GzipFile(fileobj=buf)

    esci_records = []
    for i, line in enumerate(gz):
        if i >= 30:
            break
        esci_records.append(json.loads(line))

    products = {}
    for rec in esci_records:
        for key in ['positive_passages', 'positive_passages_substitute', 'negative_passages',
                     'negative_passages_substitute', 'negative_passages_complement', 'negative_passages_true']:
            for p in rec.get(key, []):
                docid = p.get('docid', '')
                if docid and docid not in products:
                    products[docid] = p

    print(f"ESCI cargado: {len(esci_records)} queries, {len(products)} productos reales")

    docs_c = []
    for docid, p in products.items():
        title = p.get('title', '')
        desc = p.get('text', '')
        bullets = p.get('bullet_points', '')
        full_text = f"Title: {title}\n\n{desc}"
        if bullets:
            full_text += f"\n\nFeatures:\n{bullets}"
        docs_c.append(Document(
            page_content=full_text,
            metadata={"source": f"esci_{docid}.txt", "esci_id": docid}
        ))

    gt_c_raw = {}
    for rec in esci_records:
        q = rec['query']
        pos_docids = [p['docid'] for p in rec.get('positive_passages', []) if p.get('docid')]
        if pos_docids:
            gt_c_raw[q] = pos_docids

    selected_queries_c = list(gt_c_raw.keys())[:5]
    gt_c_raw = {q: gt_c_raw[q] for q in selected_queries_c}

    print(f"Queries de evaluación: {len(selected_queries_c)}")
    print(f"Documentos en corpus: {len(docs_c)}")
else:
    print("Camino C no seleccionado — saltando generación.")
```

    Camino C no seleccionado — saltando generación.

#### Ejercicio 1 — Estrategia de chunking (2 ptos)

Describe qué estrategia de chunking eliges (chunk por párrafos
vs. tamaño fijo con solapamiento) y por qué es adecuada para productos
reales de Amazon del benchmark ESCI. Justifica tu respuesta en términos
de:

- Coherencia semántica de los chunks
- Capacidad de recuperar información con consultas sobre productos
  específicos (ej. “auriculares con cancelación de ruido”, “tasa de
  retorno de calzado”)
- Ventajas del solapamiento entre chunks en este dominio

Asigna tu respuesta a `eval1_estrategia`.

``` python
if CAMINO == "C":
    eval1_estrategia = """Escribir aquí la justificación. Por ejemplo:

    Elijo chunking por tamaño fijo con RecursiveCharacterTextSplitter (chunk_size=300, overlap=50)
    porque los productos de ESCI tienen descripciones y bullet points con información detallada.
    El solapamiento evita que características concretas como "cancelación de ruido activa" o
    "batería de 12 horas" se pierdan en los cortes. La estrategia por párrafos no es adecuada
    porque las descripciones de producto son heterogéneas en longitud y estructura."""
    print("✅ eval1_estrategia asignada.")
else:
    eval1_estrategia = None
```

#### Ejercicio 2 — Indexación y recall@k (2.5 ptos)

Genera los embeddings de los productos ESCI, construye la colección en
Chroma y evalúa con recall@k usando el ground truth derivado de las
anotaciones reales del benchmark (Exact match).

``` python
if CAMINO == "C":
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks_c = splitter.split_documents(docs_c)
    print(f"Chunks generados: {len(chunks_c)}")

    collection_c = client.get_or_create_collection(
        name="hpd1_ecommerce",
        metadata={"hnsw:space": "cosine"}
    )

    esci_to_chunk_ids = {}
    for i, chunk in enumerate(chunks_c):
        embedding = model.encode(chunk.page_content).tolist()
        esci_id = chunk.metadata.get("esci_id", "")
        cid = f"eco_{i:04d}"
        collection_c.add(
            ids=[cid],
            embeddings=[embedding],
            documents=[chunk.page_content],
            metadatas=[{"esci_id": esci_id, "idx": i}]
        )
        if esci_id not in esci_to_chunk_ids:
            esci_to_chunk_ids[esci_id] = []
        esci_to_chunk_ids[esci_id].append(cid)
    print(f"Indexación completada: {collection_c.count()} vectores")

    gt_c = {}
    for q_text, docids in gt_c_raw.items():
        chunk_ids = []
        for did in docids:
            chunk_ids.extend(esci_to_chunk_ids.get(did, []))
        gt_c[q_text] = chunk_ids
    print(f"Ground truth construido: {len(gt_c)} queries con anotaciones Exact de ESCI")

    def recall_at_k(retrieved_ids, relevant_ids, k):
        if len(relevant_ids) == 0:
            return 0.0
        top_k = retrieved_ids[:k]
        hits = sum(1 for rid in relevant_ids if rid in top_k)
        return hits / len(relevant_ids)

    def buscar(collection, query, k=5):
        qe = model.encode(query).tolist()
        return collection.query(query_embeddings=[qe], n_results=k)

    eval2_recall = {}
    for k in [1, 3, 5]:
        recalls = []
        for q, relevantes in gt_c.items():
            res = buscar(collection_c, q, k=k)
            retrieved = res["ids"][0]
            r = recall_at_k(retrieved, relevantes, k)
            recalls.append(r)
        eval2_recall[f"recall@{k}"] = round(np.mean(recalls), 4)

    print("📊 recall@k sobre ESCI corpus (ground truth real):")
    for k_str, v in eval2_recall.items():
        print(f"   {k_str}: {v:.4f}")
else:
    eval2_recall = None
```

#### Ejercicio 3 — Búsqueda semántica (1.5 ptos)

Ejecuta 3 consultas de búsqueda semántica sobre el corpus ESCI y muestra
los resultados (top-3 chunks con su distancia coseno). Las consultas son
preguntas reales de compradores de Amazon.

Asigna los resultados a `eval3_busqueda`.

``` python
if CAMINO == "C":
    consultas_c = list(gt_c.keys())[:3]

    eval3_busqueda = []
    for q in consultas_c:
        res = buscar(collection_c, q, k=3)
        resultados = []
        for i in range(len(res["ids"][0])):
            resultados.append({
                "id": res["ids"][0][i],
                "distancia": round(res["distances"][0][i], 4),
                "texto": res["documents"][0][i][:200],
                "fuente": res["metadatas"][0][i].get("esci_id", ""),
            })
        eval3_busqueda.append({"query": q, "resultados": resultados})

    for r in eval3_busqueda:
        print(f"\n🔍 {r['query']}")
        for res in r["resultados"]:
            print(f"   [{res['distancia']:.4f}] {res['texto'][:100]}...")
    print("\n✅ eval3_busqueda asignada.")
else:
    eval3_busqueda = None
```

#### Ejercicio 4 — Justificación del modelo (1.5 ptos)

Justifica por qué `intfloat/e5-small-v2` es una buena elección para este
pipeline. Debes basarte en los resultados de MTEB para las tareas
BIOSSES y STS12, y explicar cómo se relacionan con la calidad esperada
en búsqueda semántica sobre productos reales de Amazon (ESCI).

Asigna tu respuesta a `eval4_justificacion`.

``` python
if CAMINO == "C":
    eval4_justificacion = """Escribir aquí la justificación. Por ejemplo:

    e5-small-v2 obtiene un Spearman de 0.774 en STS12 (similitud textual de dominio general),
    lo que indica una buena capacidad para alinear textos narrativos como descripciones de
    productos reales de Amazon. Su rendimiento en BIOSSES (0.894) muestra robustez incluso
    con vocabulario especializado. Con solo 33M de parámetros, el modelo es lo suficientemente
    ligero para ejecutarse en Colab sin GPU, adecuado para un pipeline docente."""
    print("✅ eval4_justificacion asignada.")
else:
    eval4_justificacion = None
```

## Resumen de entregables

In [21]:
print(f"Camino seleccionado: {CAMINO}")
print(f"\nVariables de corrección:")
for var in ["eval1_estrategia", "eval2_recall", "eval3_busqueda", "eval4_justificacion"]:
    val = globals().get(var, None)
    status = "✅ definida" if val is not None else "⚠️ no definida"
    print(f"  {var}: {status}")

Camino seleccionado: A

Variables de corrección:
  eval1_estrategia: ⚠️ no definida
  eval2_recall: ⚠️ no definida
  eval3_busqueda: ⚠️ no definida
  eval4_justificacion: ⚠️ no definida

# References